##### ktor-client, dataframe, kandy, org.json.XML, org.sqlite.JDBC
* Convert XML data collected via the ktor client to JSON type and load it using DataFrame.readJson.
* Load the SQLite table using DataFrame.readSqlQuery.
* Then, join the two dataframes and create a chart using Kandy.

In [55]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [56]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [57]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-21 21:23:48, wtch_dt_end:2026-07-22 21:23:48


In [58]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [59]:
@file:DependsOn("org.json:json:20250107")

In [60]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [61]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Comparable<*>,1708,1020,0,0.260000,35,null,null,null,null,null,null,null
rtmWqChpla,Comparable<*>,1708,1089,0,,141,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,1708,1,0,,1708,null,null,,,,,
rtmWqWtchStaCd,String,1708,14,0,SEA1005,141,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,1708,1708,0,1,1,854.500000,493.201446,1,427.416667,854.500000,1281.583333,1708
rtmWqTu,Comparable<*>,1708,230,0,5,143,null,null,null,null,null,null,null
ph,Comparable<*>,1708,281,0,8.070000,34,null,null,null,null,null,null,null
rtmWqSlnty,Number,1708,1559,0,0.260000,19,20.946316,10.781429,0.000000,10.329000,26.198999,29.308001,33.294998
rtmWqCndctv,Double,1708,1587,0,52.230000,6,32.824711,16.203900,0.000000,17.919500,38.737999,44.689751,53.460999
rtmWqWtchDtlDt,String,1708,148,0,2026-07-21 21:40:00.0,14,null,null,2026-07-21 21:25:00.0,2026-07-22 02:00:00.0,2026-07-22 06:45:00.0,2026-07-22 11:15:00.0,2026-07-22 15:45:00.0


In [62]:
dfRaw.head(5)

rtmWqDoxn,rtmWqChpla,rtmWqBgalgsQy,rtmWqWtchStaCd,num,rtmWqTu,ph,rtmWqSlnty,rtmWqCndctv,rtmWqWtchDtlDt,rtmWtchWtem
7.320000,3.100000,,NEP1002,1,75,7.710000,0.252000,0.521000,2026-07-21 21:25:00.0,29.340000
5.530000,,,SEA1005,2,21,7.400000,3.043000,5.635000,2026-07-21 21:25:00.0,25.240000
6.028000,18.393999,,SEA2005,3,1,7.760000,27.653000,42.988998,2026-07-21 21:25:00.0,29.850000
6.910000,15.940000,,SEA6001,4,327,8.010000,30.736000,47.327999,2026-07-21 21:25:00.0,27.040001
0.270000,1.030000,,SEA5002,5,71,7.260000,25.377001,39.932999,2026-07-21 21:25:00.0,29.080000


In [63]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert {
    rtmWqDoxn and rtmWqChpla and rtmWqSlnty and rtmWqCndctv and rtmWtchWtem and ph
}.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%.3f", value.toFloat())
}.convert {
    rtmWqTu
}.with{
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%d", value.toInt())
}



df.schema()

rtmWqDoxn: String
rtmWqChpla: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: String
ph: String
rtmWqSlnty: String
rtmWqCndctv: String
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: String

In [64]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [65]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-21T21:25,7.320,3.100,NEP1002,75,7.710,0.252,0.521,29.340
2,2026-07-21T21:25,5.530,0,SEA1005,21,7.400,3.043,5.635,25.240
3,2026-07-21T21:25,6.028,18.394,SEA2005,1,7.760,27.653,42.989,29.850
4,2026-07-21T21:25,6.910,15.940,SEA6001,327,8.010,30.736,47.328,27.040
5,2026-07-21T21:25,0.270,1.030,SEA5002,71,7.260,25.377,39.933,29.080


In [66]:
USE {
    dependencies {
        implementation("org.xerial:sqlite-jdbc:3.49.1.0")
        implementation("ch.qos.logback:logback-classic:1.5.12")
    }
}

In [67]:
import java.sql.Connection
import java.sql.DriverManager

Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite")

In [68]:
val sqlStmt = "SELECT * FROM OWQObservatory"
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sta_code,String,19,19,0,SEA1002,1,null,null,NEP1001,NEP3001,SEA1301,SEA3003,SEA7002
sta_name,String,19,19,0,시화조력,1,null,null,광양망덕,낙동명지,부산수영,영산목포,천수만
ocean_division,String,19,2,0,특별관리해역,12,null,null,특별관리해역,특별관리해역,특별관리해역,하구 및 만,하구 및 만
lon,Double,19,19,0,126.611000,1,127.588263,1.113453,126.366000,126.540167,127.605000,128.615167,129.387000
lat,Double,19,18,0,35.802000,2,35.687263,0.981615,34.782000,34.990667,35.211000,35.947833,37.731000


In [69]:
val joinedDf = removedDf.join(df_list) { 관측정점코드 match right.sta_code }

In [73]:
joinedDf
    .select{  일시 and 클로로필 and sta_name   }
    .convert{클로로필}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(클로로필) {axis.name ="클로로필"}
        line{
            color(sta_name){
             //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="pc2LpN" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("pc2LpN");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"sta_name":["낙동명지","시화반월","마산봉암","새만금","광양초남","광양망덕","금강하구","영산목포","울산매암","광양적량","천수만","광양망덕","낙동명지","광양초남","금강하구","울산매암","광양적량","새만금","영산목포","마산봉암","천수만","영산목포","울산매암","마산봉암","광양적량","금강하구","새만금","낙동명지","광양망덕","시화조력","시화반월","영산영암","광양초남","영산영암","시화반월","광양초남","새만금","광양망덕","금강하구","울산매암","낙동명지","낙동명지","광양망덕","금강하구","광양적량","시화반월","영산영암","울산매암","천수만","광양초남","새만금","마산봉암","금강하구","광양적량","낙동명지","시화반월","광양초남","마산봉암","영산영암","광양망덕","광양적량","광양망덕","영산영암","낙동명지","천수만","광양초남","새만금","시화조력","금강하구","시화반월","마산봉암","울산매암","영산목포","영산목포","광양망덕","울산매암","마산봉암","낙동명지","영산영암","새만금","광양초남","광양적량","시화반월","금강하구","새만금","시화반월","금강하구","천수만","영산영암","광양망덕","마산봉암","영산목포","광양초남","울산매암","광양적량","시화조력","낙동명지","시화반월","낙동명지","금강하구","영산목포","광양망덕","마산봉암","광양초남","영산영암","울산매암","영산영암","광양초남","마산봉암","천수만","새만금","광양적량","금강하구","시화반월","낙동명지","영산목포","시화조력","광양망덕","금강하구","마산봉암","영산목포","광양적량","낙동명지","시화반월","영산영암","광양초남","새만금","광양적량","마산봉암","광양초남","광양망덕","새만금","시화반월","울산매암","영산목포","영산영암","시화조력","금강하구","영산목포","광양적량","울산매암","시화반월","새만금","광양초남","광양망덕","영산영암","마산봉암","울산매암","낙동명지","새만금","광양적량","영산영암","광양망덕","마산봉암","천수만","시화반월","영산목포","시화조력","금강하구","마산봉암","광양적량","새만금","영산목포","시화반월","울산매암","영산영암","낙동명지","금강하구","금강하구","마산봉암","시화조력","낙동명지","광양망덕","새만금","영산영암","시화반월","영산목포","천수만","울산매암","광양적량","광양초남","광양망덕","광양적량","낙동명지","영산영암","금강하구","마산봉암","새만금","시화반월","광양초남","영산목포","울산매암","영산영암","금강하구","광양망덕","광양초남","시화조력","광양적량","영산목포","새만금","시화반월","낙동명지","울산매암","광양초남","금강하구","영산목포","새만금","광양적량","시화반월","광양망덕","금강하구","광양망덕","광양적량","영산목포","천수만","낙동명지","영산영암","울산매암","새만금","시화반월","시화조력","광양초남","영산영암","영산목포","시화반월","울산매암","새만금","광양망덕","광양초남","낙동명지","광양적량","천수만","시화조력","낙동명지","금강하구","영산영암","광양망덕","울산매암","광양초남","광양적량","시화반월","영산목포","새만금","낙동명지","영산영암","광양적량","시화반월","새만금","울산매암","영산목포","금강하구","광양망덕","울산매암","영산목포","시화반월","새만금","광양망덕","낙동명지","영산영암","광양초남","금강하구","광양적량","시화조력","천수만","울산매암","시화반월","광양적량","영산목포","새만금","광양초남","낙동명지","영산영암","광양망덕","시화조력","천수만","울산매암","광양초남","새만금","낙동명지","금강하구","영산영암","광양망덕","광양적량","시화반월","영산목포","낙동명지","광양망덕","울산매암","새만금","영산목포","금강하구","광양적량","시화반월","새만금","금강하구","영산영암","영산목포","광양망덕","시화반월","천수만","울산매암","광양초남","시화조력","낙동명지","광양적량","광양적량","낙동명지","금강하구","시화반월","광양망덕","새만금","영산목포","울산매암","광양초남","광양망덕","낙동명지","영산영암","영산목포","금강하구","시화조력","시화반월","천수만","광양초남","광양적량","새만금","울산매암","영산영암","낙동명지","울산매암","시화반월","광양적량","금강하구","새만금","광양망덕","광양적량","새만금","천수만","광양초남","낙동명지","금강하구","영산영암","영산목포","시화조력","울산매암","광양망덕","시화반월","낙동명지","새만금","영산영암","광양망덕","영산목포","광양초남","울산매암","금강하구","광양적량","시화반월","영산영암","천수만","광양초남","시화반월","광양적량","새만금","금강하구","울산매암","광양망덕","낙동명지","영산목포","시화조력","광양초남","광양적량","광양망덕","낙동명지","새만금","울산매암","금강하구","시화반월","시화조력","울산매암","영산영암","광양적량","천수만","광양망덕","금강하구","시화반월","광양초남","낙동명지","새만금","영산목포","광양적량","울산매암","광양초남","시화반월","영산목포","낙동명지","금강하구","광양망덕","영산영암","새만금","천수만","영산목포","울산매암","광양망덕","낙동명지","시화조력","새만금","영산영암","금강하구","시화반월","광양적량","광양초남","울산매암","광양망덕","광양적량","낙동명지","광양초남","새만금","영산목포","금강하구","영산영암","시화반월","광양초남","천수만","영산목포","금강하구","새만금","울산매암","낙동명지","시화조력","영산영암","광양망덕","광양적량","시화반월","울산매암","낙동명지","광양적량","새만금","금강하구","